# MNIST Vision Transformer on 2x T4 (PyTorch counterpart)

This notebook runs the PyTorch counterpart of the hand-written CUDA ViT. It is
set up for comparison against `kaggle_run.ipynb`:

- same Kaggle CSV search and torchvision MNIST fallback;
- same model config: V=256, T=784, L=2, D=64, H=4, C=10;
- same train runs: Adam lr=1e-3, 3000 steps, per-rank B=32;
- same log columns and throughput timer boundaries.

The PyTorch script is written in the normal ML style: `nn.Module` layers,
`DataLoader`, `torch.optim.Adam`, and `DistributedDataParallel` for two GPUs.
The architecture and experiment settings stay aligned with the CUDA notebook.
Attention is pinned to PyTorch's SDPA math backend in the source so the run
does not depend on automatic Flash or memory-efficient backend selection.

For throughput, use the same Kaggle accelerator and the same CSV in both
notebooks. Loss/accuracy curves are a training-behavior comparison, not an
exact numeric parity test: C++ and PyTorch RNG streams do not give identical
initial weights or sampled batches from seed 42.

Kaggle setup:

1. Select `GPU T4 x2` in Settings.
2. Enable Internet only if the MNIST fallback must download data.
3. Add the Digit Recognizer dataset when comparing with the CUDA notebook.


## 1. Verify the environment

Expect two T4 GPUs and a CUDA PyTorch build.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda devices:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'


## 2. Write the PyTorch source

The source is embedded so this notebook can run without uploading the repo.

In [ ]:
%%writefile train_vit_torch.py
#!/usr/bin/env python3
"""Idiomatic PyTorch training path for the MNIST pixel-token ViT."""

from __future__ import annotations

import argparse
import csv
import os
import time
from array import array
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch import nn
from torch.nn.attention import SDPBackend, sdpa_kernel
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset, RandomSampler


@dataclass(frozen=True)
class Cfg:
    vocab: int = 256
    seq: int = 28 * 28
    layers: int = 2
    dim: int = 64
    heads: int = 4
    classes: int = 10


DEFAULT_PARAM_COUNT = 167_296


class MnistCsvDataset(Dataset):
    def __init__(self, path: Path, seq: int) -> None:
        self.pixels, self.labels = load_mnist_csv(path, seq)

    def __len__(self) -> int:
        return self.labels.numel()

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.pixels[index], self.labels[index]


class TransformerBlock(nn.Module):
    def __init__(self, cfg: Cfg) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.dim, eps=1e-5)
        self.attn = nn.MultiheadAttention(
            embed_dim=cfg.dim,
            num_heads=cfg.heads,
            dropout=0.0,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(cfg.dim, eps=1e-5)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.dim, 4 * cfg.dim),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * cfg.dim, cfg.dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm1(x)
        with sdpa_kernel(SDPBackend.MATH):
            attn_out = self.attn(h, h, h, need_weights=False)[0]
        x = x + attn_out
        return x + self.mlp(self.norm2(x))


class PixelViT(nn.Module):
    def __init__(self, cfg: Cfg) -> None:
        super().__init__()
        if cfg.dim % cfg.heads:
            raise ValueError("dim must be divisible by heads")

        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab, cfg.dim)
        self.position_embedding = nn.Parameter(torch.empty(1, cfg.seq, cfg.dim))
        self.blocks = nn.ModuleList(TransformerBlock(cfg) for _ in range(cfg.layers))
        self.norm = nn.LayerNorm(cfg.dim, eps=1e-5)
        self.head = nn.Linear(cfg.dim, cfg.classes, bias=False)
        self.apply(init_module)
        nn.init.normal_(self.position_embedding, mean=0.0, std=0.02)

    def forward(self, pixels: torch.Tensor) -> torch.Tensor:
        if pixels.shape[1] != self.cfg.seq:
            raise ValueError(f"expected sequence length {self.cfg.seq}, got {pixels.shape[1]}")

        x = self.token_embedding(pixels) + self.position_embedding
        for block in self.blocks:
            x = block(x)
        x = self.norm(x).mean(dim=1)
        return self.head(x)


def init_module(module: nn.Module) -> None:
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if getattr(module, "bias", None) is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.LayerNorm):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)
    elif isinstance(module, nn.MultiheadAttention):
        nn.init.normal_(module.in_proj_weight, mean=0.0, std=0.02)
        if module.in_proj_bias is not None:
            nn.init.zeros_(module.in_proj_bias)


def load_mnist_csv(path: Path, seq: int) -> tuple[torch.Tensor, torch.Tensor]:
    labels = array("q")
    pixels_raw = bytearray()
    with path.open(newline="") as fp:
        rows = csv.reader(fp)
        header = next(rows, None)
        if header is None or len(header) != seq + 1:
            raise ValueError(f"{path} must have label + {seq} pixel columns")

        for line_no, row in enumerate(rows, start=2):
            if len(row) != seq + 1:
                raise ValueError(f"{path}:{line_no} has {len(row)} columns")
            labels.append(int(row[0]))
            pixels_raw.extend(int(pixel) for pixel in row[1:])

    if not labels:
        raise ValueError(f"{path} contains no training samples")

    labels_tensor = torch.frombuffer(labels, dtype=torch.int64).clone()
    pixels_tensor = torch.frombuffer(pixels_raw, dtype=torch.uint8).clone().view(-1, seq)
    return pixels_tensor, labels_tensor


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


def distributed_context(device_arg: str) -> tuple[int, int, torch.device]:
    world = int(os.environ.get("WORLD_SIZE", "1"))
    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", str(rank)))

    if device_arg == "auto":
        device_arg = "cuda" if torch.cuda.is_available() else "cpu"
    if device_arg == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("--device cuda requested but CUDA is unavailable")
        device = torch.device("cuda", local_rank % torch.cuda.device_count())
        torch.cuda.set_device(device)
    else:
        device = torch.device(device_arg)

    if world > 1:
        dist.init_process_group(backend="nccl" if device.type == "cuda" else "gloo")
    return rank, world, device


def reduce_metrics(
    loss: torch.Tensor,
    logits: torch.Tensor,
    labels: torch.Tensor,
    world: int,
) -> tuple[float, float]:
    with torch.no_grad():
        count = torch.tensor(labels.numel(), device=labels.device, dtype=torch.float32)
        loss_sum = loss.detach() * count
        correct = (logits.argmax(dim=1) == labels).sum(dtype=torch.float32)
        if world > 1:
            dist.all_reduce(loss_sum, op=dist.ReduceOp.SUM)
            dist.all_reduce(correct, op=dist.ReduceOp.SUM)
            dist.all_reduce(count, op=dist.ReduceOp.SUM)
        return (loss_sum / count).item(), (correct / count).item()


def sync_device(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="PyTorch MNIST pixel-token ViT counterpart for train_vit.cu.",
    )
    parser.add_argument("csv_path", nargs="?", default="data/train.csv")
    parser.add_argument("steps", nargs="?", type=int, default=200)
    parser.add_argument("batch_size", nargs="?", type=int, default=8)
    parser.add_argument("lr", nargs="?", type=float, default=0.05)
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    parser.add_argument("--log-path", default="training_log.csv")
    parser.add_argument("--log-every", type=int, default=10)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--seed", type=int, default=42)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    cfg = Cfg()
    rank, world, device = distributed_context(args.device)
    log_fp = None

    try:
        torch.manual_seed(args.seed)
        dataset = MnistCsvDataset(Path(args.csv_path), cfg.seq)
        sampler_rng = torch.Generator()
        sampler_rng.manual_seed(args.seed + 1000 * rank)
        sampler = RandomSampler(
            dataset,
            replacement=True,
            num_samples=args.steps * args.batch_size,
            generator=sampler_rng,
        )
        loader = DataLoader(
            dataset,
            batch_size=args.batch_size,
            sampler=sampler,
            num_workers=args.num_workers,
            pin_memory=device.type == "cuda",
            drop_last=True,
        )

        model = PixelViT(cfg).to(device)
        param_count = count_parameters(model)
        if param_count != DEFAULT_PARAM_COUNT:
            raise AssertionError(f"default model parameter count drifted: {param_count}")
        train_model: nn.Module
        if world > 1:
            train_model = DDP(
                model,
                device_ids=[device.index] if device.type == "cuda" else None,
            )
        else:
            train_model = model
        optimizer = torch.optim.Adam(
            train_model.parameters(),
            lr=args.lr,
            betas=(0.9, 0.999),
            eps=1e-8,
        )

        if rank == 0:
            print(
                f"ranks={world} N={len(dataset)} B={args.batch_size} "
                f"T={cfg.seq} L={cfg.layers} D={cfg.dim} H={cfg.heads} "
                f"C={cfg.classes} params={param_count} "
                f"steps={args.steps} lr={args.lr:g} impl=torch device={device.type}",
                flush=True,
            )
            log_fp = Path(args.log_path).open("w", newline="")
            log_writer = csv.writer(log_fp)
            log_writer.writerow(("step", "elapsed_s", "loss", "accuracy"))
            log_fp.flush()
        else:
            log_writer = None

        sync_device(device)
        start = time.perf_counter()
        train_model.train()

        for step, (pixels, labels) in enumerate(loader, start=1):
            pixels = pixels.to(device=device, dtype=torch.long, non_blocking=True)
            labels = labels.to(device=device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = train_model(pixels)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            should_log = step % args.log_every == 0 or step == args.steps
            if should_log:
                mean_loss, accuracy = reduce_metrics(loss, logits, labels, world)
                if rank == 0:
                    sync_device(device)
                    elapsed = time.perf_counter() - start
                    print(f"step {step:4d} | loss {mean_loss:.4f} | acc {accuracy:.3f}", flush=True)
                    if log_writer is not None and log_fp is not None:
                        log_writer.writerow(
                            (step, f"{elapsed:.2f}", f"{mean_loss:.4f}", f"{accuracy:.4f}")
                        )
                        log_fp.flush()

        sync_device(device)
        if world > 1:
            dist.barrier()
        elapsed = time.perf_counter() - start
        total_images = args.steps * args.batch_size * world
        if rank == 0:
            print(f"\nfinished: {args.steps} steps in {elapsed:.2f}s")
            print(
                f"throughput: {total_images / elapsed:.0f} img/s global  |  "
                f"{total_images / elapsed / world:.0f} img/s/GPU"
            )
    finally:
        if log_fp is not None:
            log_fp.close()
        if dist.is_initialized():
            dist.destroy_process_group()


if __name__ == "__main__":
    main()


## 3. Check the model contract

The default PyTorch model should keep the CUDA-comparable parameter count.

In [ ]:
from train_vit_torch import Cfg, DEFAULT_PARAM_COUNT, PixelViT, count_parameters
cfg = Cfg()
model = PixelViT(cfg)
print(cfg)
print('parameter count:', count_parameters(model))
assert count_parameters(model) == DEFAULT_PARAM_COUNT


## 4. Locate or materialize the data

This is the same CSV search order and MNIST fallback as the CUDA notebook.

In [ ]:
import glob
import os
import numpy as np

known = [
    '/kaggle/input/digit-recognizer/train.csv',
    '/kaggle/input/fashionmnist/fashion-mnist_train.csv',
    '/kaggle/input/fashion-mnist/fashion-mnist_train.csv',
]
globbed = sorted(set(
    glob.glob('/kaggle/input/**/*train*.csv', recursive=True) +
    glob.glob('/kaggle/input/**/*Train*.csv', recursive=True)))

def valid_csv(path):
    try:
        with open(path) as f:
            return len(f.readline().split(',')) == 785
    except Exception:
        return False

CSV = next((c for c in known + globbed if os.path.exists(c) and valid_csv(c)), None)

if CSV is None:
    print('Falling back to torchvision MNIST...')
    from torchvision.datasets import MNIST
    ds = MNIST(root='/kaggle/working/mnist_raw', train=True, download=True)
    labels = ds.targets.numpy().astype(np.int32)
    pixels = ds.data.numpy().reshape(-1, 784).astype(np.int32)
    arr = np.concatenate([labels[:, None], pixels], axis=1)
    header = 'label,' + ','.join(f'pixel{i}' for i in range(784))
    CSV = '/kaggle/working/train.csv'
    np.savetxt(CSV, arr, fmt='%d', delimiter=',', header=header, comments='')

assert os.path.exists(CSV), CSV
print('Using CSV:', CSV)
print('Size     :', os.path.getsize(CSV) // (1024 * 1024), 'MiB')
os.environ['CSV'] = CSV
!head -c 120 "$CSV" ; echo
!wc -l "$CSV"


## 5. Training smoke test

Run one normal PyTorch step before profiling. This leaves the real training traceback visible if the embedded source or CUDA environment is stale.

In [ ]:
!python train_vit_torch.py "$CSV" 1 8 0.001 \
  --device cuda --log-path torch_smoke_log.csv


## 6. GPU, memory, and DDP profiling

Runs 50 PyTorch steps for B in {8, 16, 32, 64} on one GPU, plus one two-GPU run at B=32. The internal throughput timer matches the training script timer and excludes model/data setup.

In [ ]:
import os
import subprocess
import threading
import time
import numpy as np

try:
    import psutil
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'psutil'], check=True)
    import psutil

def _gpu_monitor(stop_evt, records, interval=0.25):
    while not stop_evt.is_set():
        r = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=index,utilization.gpu,memory.used,memory.total,power.draw',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True)
        ts = time.time()
        for line in r.stdout.strip().splitlines():
            parts = [x.strip() for x in line.split(',')]
            if len(parts) < 5:
                continue
            try:
                records.append(dict(
                    ts=ts, gpu=int(parts[0]),
                    gpu_util=float(parts[1]),
                    mem_mb=float(parts[2]),
                    mem_total=float(parts[3]),
                    power=float(parts[4]) if 'N/A' not in parts[4] else 0.0,
                ))
            except ValueError:
                pass
        time.sleep(interval)

def run_prof(csv_path, B, steps=50, lr=0.001, np_=1):
    if np_ == 1:
        cmd = ['python', 'train_vit_torch.py', csv_path, str(steps), str(B), str(lr),
               '--device', 'cuda', '--log-path', 'torch_profile_log.csv']
    else:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={np_}',
               '--master_port=29531', 'train_vit_torch.py',
               csv_path, str(steps), str(B), str(lr),
               '--device', 'cuda', '--log-path', 'torch_profile_log.csv']
    stop = threading.Event()
    recs, cpu_s = [], []

    def _cpu():
        while not stop.is_set():
            cpu_s.append((time.time(), psutil.cpu_percent()))
            time.sleep(0.25)

    gmon = threading.Thread(target=_gpu_monitor, args=(stop, recs), daemon=True)
    cmon = threading.Thread(target=_cpu, daemon=True)
    gmon.start()
    cmon.start()

    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0

    stop.set()
    time.sleep(0.4)
    if proc.returncode:
        raise RuntimeError(
            f"profile run failed: {cmd}\n"
            f"--- stdout ---\n{proc.stdout}\n"
            f"--- stderr ---\n{proc.stderr}"
        )

    tput = None
    for line in proc.stdout.splitlines():
        if 'img/s' in line and 'throughput' in line.lower():
            for tok in line.replace('|', ' ').split():
                try:
                    tput = float(tok)
                    break
                except ValueError:
                    pass
            break

    return dict(recs=recs, cpu=cpu_s, elapsed=elapsed,
                tput=tput, stdout=proc.stdout)

CSV = os.environ.get('CSV', '')
assert CSV, 'Run the data cell first'
n_gpus = int(subprocess.check_output(
    'nvidia-smi --query-gpu=name --format=csv,noheader | wc -l',
    shell=True, text=True).strip())
print(f'GPUs detected: {n_gpus}')

prof = {}
BATCH_SIZES = [8, 16, 32, 64]
for B in BATCH_SIZES:
    print(f'  1-GPU  B={B:3d} / 50 steps ...', end=' ', flush=True)
    prof[('1gpu', B)] = run_prof(CSV, B, steps=50, np_=1)
    print(f"done  {prof[('1gpu', B)]['tput']} img/s  "
          f"elapsed={prof[('1gpu', B)]['elapsed']:.1f}s")

if n_gpus >= 2:
    print('  2-GPU  B= 32 / 50 steps (PyTorch DDP) ...',
          end=' ', flush=True)
    prof[('2gpu', 32)] = run_prof(CSV, 32, steps=50, np_=2)
    print(f"done  {prof[('2gpu', 32)]['tput']} img/s  "
          f"elapsed={prof[('2gpu', 32)]['elapsed']:.1f}s")
else:
    print('Single-GPU session - 2-GPU profiling skipped.')


## 6b. Plot profiling results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BATCH_SIZES = [8, 16, 32, 64]
cmap = plt.cm.tab10(np.linspace(0, 0.8, len(BATCH_SIZES)))

def gpu_ts(recs, gpu_idx=0):
    rows = [r for r in recs if r['gpu'] == gpu_idx]
    if not rows:
        return [], [], [], []
    t0 = rows[0]['ts']
    return (
        [r['ts'] - t0 for r in rows],
        [r['gpu_util'] for r in rows],
        [r['mem_mb'] / 1024 for r in rows],
        [r['power'] for r in rows],
    )

fig1, axes = plt.subplots(4, len(BATCH_SIZES), figsize=(5 * len(BATCH_SIZES), 12))
fig1.suptitle('PyTorch GPU and CPU time-series - 50 steps, 1 GPU',
              fontsize=13, fontweight='bold')

for idx, (B, color) in enumerate(zip(BATCH_SIZES, cmap)):
    exp = prof.get(('1gpu', B))
    if not exp:
        for row in range(4):
            axes[row][idx].set_visible(False)
        continue
    ts, util, mem, pwr = gpu_ts(exp['recs'], 0)
    for row, (values, label) in enumerate([
        (util, 'GPU util %'),
        (mem, 'GPU mem GiB'),
        (pwr, 'GPU power W'),
    ]):
        ax = axes[row][idx]
        ax.plot(ts, values, color=color, linewidth=1.3)
        ax.fill_between(ts, values, alpha=0.12, color=color)
        ax.set_title(f'B={B}', fontsize=10)
        ax.set_xlabel('Time (s)', fontsize=8)
        ax.set_ylabel(label, fontsize=8)
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=0)

    ax_cpu = axes[3][idx]
    if exp['cpu']:
        ct0 = exp['cpu'][0][0]
        ax_cpu.plot([x[0] - ct0 for x in exp['cpu']],
                    [x[1] for x in exp['cpu']], color=color, linewidth=1.3)
    ax_cpu.set_title(f'B={B}', fontsize=10)
    ax_cpu.set_xlabel('Time (s)', fontsize=8)
    ax_cpu.set_ylabel('CPU util %', fontsize=8)
    ax_cpu.grid(alpha=0.3)
    ax_cpu.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('torch_profiling_timeseries.png', dpi=130, bbox_inches='tight')
plt.show()

fig2, axes2 = plt.subplots(2, 2, figsize=(12, 8))
fig2.suptitle('PyTorch summary metrics vs batch size - 50 steps, 1 GPU',
              fontsize=13, fontweight='bold')

tputs = [prof.get(('1gpu', B), {}).get('tput') or 0 for B in BATCH_SIZES]
peak_mem = [max((r['mem_mb'] for r in prof.get(('1gpu', B), {}).get('recs', [])
                 if r['gpu'] == 0), default=0) / 1024 for B in BATCH_SIZES]
avg_util, avg_cpu = [], []
for B in BATCH_SIZES:
    recs = prof.get(('1gpu', B), {}).get('recs', [])
    cpu = prof.get(('1gpu', B), {}).get('cpu', [])
    gpu_values = [r['gpu_util'] for r in recs if r['gpu'] == 0]
    avg_util.append(np.mean(gpu_values) if gpu_values else 0)
    avg_cpu.append(np.mean([v for _, v in cpu]) if cpu else 0)

labels = [f'B={B}' for B in BATCH_SIZES]
for ax, values, ylabel, title in [
    (axes2[0, 0], tputs, 'img/s', 'Throughput'),
    (axes2[0, 1], peak_mem, 'GiB', 'Peak GPU memory'),
    (axes2[1, 0], avg_util, '%', 'Average GPU utilization'),
    (axes2[1, 1], avg_cpu, '%', 'Average CPU utilization'),
]:
    bars = ax.bar(labels, values, color=cmap, edgecolor='white', linewidth=0.5)
    ax.bar_label(bars, fmt='%.1f', fontsize=9, padding=2)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('torch_profiling_summary.png', dpi=130, bbox_inches='tight')
plt.show()

if ('2gpu', 32) in prof:
    fig3, (ax_u, ax_m) = plt.subplots(1, 2, figsize=(14, 5))
    fig3.suptitle('PyTorch 1 GPU vs 2 GPU at B=32', fontsize=13, fontweight='bold')
    ts1, u1, m1, _ = gpu_ts(prof[('1gpu', 32)]['recs'], 0)
    ts2a, u2a, m2a, _ = gpu_ts(prof[('2gpu', 32)]['recs'], 0)
    ts2b, u2b, m2b, _ = gpu_ts(prof[('2gpu', 32)]['recs'], 1)
    for ax, one, two_a, two_b, ylabel in [
        (ax_u, u1, u2a, u2b, 'GPU util (%)'),
        (ax_m, m1, m2a, m2b, 'GPU mem (GiB)'),
    ]:
        ax.plot(ts1, one, 'b-', linewidth=1.8, label='1 GPU, GPU0')
        ax.plot(ts2a, two_a, 'r-', linewidth=1.8, label='2 GPU, GPU0')
        ax.plot(ts2b, two_b, 'r--', linewidth=1.3, label='2 GPU, GPU1')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(alpha=0.3)
        ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.savefig('torch_profiling_allreduce.png', dpi=130, bbox_inches='tight')
    plt.show()
else:
    print('No 2-GPU profiling data to plot.')


## 7. Single-GPU training run

Matches the CUDA notebook run: Adam lr=1e-3, 3000 steps, per-rank B=32.

In [ ]:
!python train_vit_torch.py "$CSV" 3000 32 0.001 \
  --device cuda --log-path training_log_torch.csv


## 8. Save the single-GPU PyTorch log

In [ ]:
import os, shutil
shutil.copy('training_log_torch.csv', 'torch_1gpu_log.csv')
print('Saved torch_1gpu_log.csv:', os.path.getsize('torch_1gpu_log.csv'), 'bytes')


## 9. Two-GPU training run

`torchrun` launches one rank per GPU. Per-rank B=32 means global B=64, matching the CUDA notebook multi-GPU run.

In [ ]:
!torchrun --standalone --nproc_per_node=2 --master_port=29541 \
  train_vit_torch.py "$CSV" 3000 32 0.001 \
  --device cuda --log-path training_log_torch.csv


## 10. Save the two-GPU PyTorch log

In [ ]:
import os, shutil
shutil.copy('training_log_torch.csv', 'torch_2gpu_log.csv')
print('Saved torch_2gpu_log.csv:', os.path.getsize('torch_2gpu_log.csv'), 'bytes')


## 11. PyTorch loss and accuracy vs time

The x-axis is wall-clock time, so this is the scaling view.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

logs = {'PyTorch 1 GPU': 'torch_1gpu_log.csv',
        'PyTorch 2 GPU': 'torch_2gpu_log.csv'}
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))

for label, path in logs.items():
    if not os.path.exists(path):
        print('Missing:', path)
        continue
    df = pd.read_csv(path)
    smooth = lambda s: s.rolling(window=10, min_periods=1).mean()
    line, = ax_loss.plot(df['elapsed_s'], smooth(df['loss']),
                         label=label, linewidth=2)
    ax_loss.plot(df['elapsed_s'], df['loss'], alpha=0.15, color=line.get_color())
    line, = ax_acc.plot(df['elapsed_s'], smooth(df['accuracy']),
                        label=label, linewidth=2)
    ax_acc.plot(df['elapsed_s'], df['accuracy'], alpha=0.15, color=line.get_color())

ax_loss.set_xlabel('Wall-clock time, seconds')
ax_loss.set_ylabel('Cross-entropy loss')
ax_loss.set_title('Loss vs time')
ax_loss.legend()
ax_loss.grid(alpha=0.3)
ax_acc.set_xlabel('Wall-clock time, seconds')
ax_acc.set_ylabel('Accuracy')
ax_acc.set_title('Accuracy vs time')
ax_acc.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax_acc.legend()
ax_acc.grid(alpha=0.3)
plt.suptitle('PyTorch 1 GPU vs 2 GPU - Adam lr=1e-3, per-rank B=32, 3000 steps',
             fontsize=13)
plt.tight_layout()
plt.savefig('torch_training_curves.png', dpi=150)
plt.show()


## 12. Optional CUDA vs PyTorch plot

Copy CUDA notebook logs into this session as:

- `cuda_1gpu_log.csv`
- `cuda_2gpu_log.csv`

The plot expects the same CSV dataset and the same run arguments. Compare
throughput and time-to-loss trends. Do not interpret point-by-point loss
differences as a kernel parity failure unless both implementations are fed the
same parameters and sampled batches.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

pairs = [
    ('1 GPU', 'cuda_1gpu_log.csv', 'torch_1gpu_log.csv'),
    ('2 GPU', 'cuda_2gpu_log.csv', 'torch_2gpu_log.csv'),
]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for gpu_label, cuda_path, torch_path in pairs:
    for impl, path, style in [('CUDA', cuda_path, '-'), ('PyTorch', torch_path, '--')]:
        if not os.path.exists(path):
            print('Missing optional comparison log:', path)
            continue
        df = pd.read_csv(path)
        axes[0].plot(df['elapsed_s'], df['loss'].rolling(10, min_periods=1).mean(),
                     linestyle=style, linewidth=2, label=f'{impl} {gpu_label}')
        axes[1].plot(df['elapsed_s'],
                     df['accuracy'].rolling(10, min_periods=1).mean(),
                     linestyle=style, linewidth=2, label=f'{impl} {gpu_label}')

axes[0].set_xlabel('Wall-clock time, seconds')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Training loss')
axes[1].set_xlabel('Wall-clock time, seconds')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training accuracy')
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.suptitle('CUDA vs PyTorch logs with matching run settings', fontsize=13)
plt.tight_layout()
plt.show()


## Comparison notes

- The CUDA notebook and this notebook use the same data locator and fallback
  materialization to the Kaggle CSV format.
- The reported global batch is not the same between 1 GPU and 2 GPU runs:
  both notebooks use per-rank B=32, so two GPUs use global B=64.
- The PyTorch code uses standard `nn.Module`, `DataLoader`, Adam, and DDP
  rather than copying CUDA memory layout and kernel boundaries.
- PyTorch attention is pinned with `sdpa_kernel(SDPBackend.MATH)` for these
  comparisons.
- For a strict numerical forward/backward parity test, add a shared parameter
  fixture and a shared batch-index fixture to both implementations first.
